# ROSMAP DLPFC Covariate preprocessing 

Generate the Genotype process data for Covariate data. (The Genotype data for association has been performed in [pQTL analysis](https://github.com/cumc/fungen-xqtl-analysis/blob/main/analysis/Wang_Columbia/ROSMAP/pqtl/genotype_preprocessing.ipynb),  (Once for all tissues) )

- `input`:    
    1. genotype data `/mnt/vast/hpc/csg/molecular_phenotype_calling/genotype/ROSMAP_NIA_WGS.leftnorm.filtered.filtered`
    2. rnaSeq id lookup file `ROSMAP_JointCall_sample_participant_lookup_fixed.rnaseq`
    3. phenotype data from previous data. 

- `output`:
    1. the covariate data for association analysis


In [ ]:
### Genotype processing

# step 4:

cd ~/Work/leaf_cutter2/ROSMAP_DLPFC_2024/

sos run pipeline/GWAS_QC.ipynb genotype_phenotype_sample_overlap \
        --cwd output/data_preprocessing/genotype_data/ \
        --phenoFile output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.formated.bed.gz  \
        --genoFile Genotype/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.fam  \
        --container /mnt/vast/hpc/csg/containers_xqtl/bioinfo.sif \
        --mem 40G -s force

# step 5:kinship

sos run pipeline/GWAS_QC.ipynb king \
    --cwd output/data_preprocessing/genotype_data  \
    --genoFile Genotype/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.bed \
    --name ROSMAP_DLPFC \
    --keep-samples output/data_preprocessing/genotype_data/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.formated.bed.sample_genotypes.txt \
    --container /mnt/vast/hpc/csg/containers_xqtl/bioinfo.sif --no-maximize-unrelated \
    --mem 80G -s force

# step 6:qc, no related samples

sos run pipeline/GWAS_QC.ipynb qc \
   --cwd cache/geno_data \
   --genoFile Genotype/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.bed \
   --keep-samples output/data_preprocessing/genotype_data/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.formated.bed.sample_genotypes.txt \
   --mac-filter 5 \
   --container /mnt/vast/hpc/csg/containers_xqtl/bioinfo.sif \
   --mem 60G -s force

# step 7: pca

#### PCA on genotype---pca
sos run pipeline/PCA.ipynb flashpca \
   --cwd output/data_preprocessing/genotype_data/PCA \
   --genoFile cache/geno_data/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.plink_qc.prune.bed \
   --container /mnt/vast/hpc/csg/containers_xqtl/flashpcaR.sif \
   --mem 60G -s force

# Step 8: covariate -- all the tables (tsv files) actually are same for all tissues, so we simply just
# copy one and distribute to different folders

sos run pipeline/covariate_formatting.ipynb merge_genotype_pc \
    --cwd output/data_preprocessing/covariate_data \
    --pcaFile output/data_preprocessing/genotype_data/PCA/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.plink_qc.prune.pca.rds \
    --covFile  /mnt/vast/hpc/csg/wanggroup/fungen-xqtl-analysis/analysis/Wang_Columbia/ROSMAP/pseudo_bulk_eqtl_kelli/Inh/output/data_preprocessing/covariate_data/rosmap_cov.txt \
    --tol_cov 0.4  \
    --k 15 \
    --container /mnt/vast/hpc/csg/containers_xqtl/bioinfo.sif --mem 20G -s force

# step 9:

# Hidden factor - updated updated
sos run pipeline/covariate_hidden_factor.ipynb Marchenko_PC \
   --cwd output/data_preprocessing/covariate_data \
   --phenoFile output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.formated.bed.gz  \
   --covFile output/data_preprocessing/covariate_data/rosmap_cov.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.plink_qc.prune.pca.gz \
   --container /mnt/vast/hpc/csg/containers_xqtl/PCAtools.sif --mem 80G -s force